# GraphRAG Indexing Evaluation

Este notebook executa a indexação completa do GraphRAG em Neo4j.

Fluxo:
1. Configuração de caminhos e parâmetros.
2. Limpeza total de dados já existentes no banco.
3. Execução do pipeline de indexação (load/chunk -> extract -> communities -> reports -> embed).
4. Verificação final de contagens no grafo.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# Ajuste automático de caminho para conseguir importar o pacote local.
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
GRAPHRAG_ROOT = REPO_ROOT / "graphrag-neo4j-langchain"
SRC_PATH = GRAPHRAG_ROOT / "src"

if str(GRAPHRAG_ROOT) not in sys.path:
    sys.path.insert(0, str(GRAPHRAG_ROOT))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Carrega variáveis de ambiente do projeto GraphRAG, se existir .env.
env_path = GRAPHRAG_ROOT / ".env"
if env_path.exists():
    try:
        from dotenv import load_dotenv
        load_dotenv(env_path)
        print(f".env carregado de: {env_path}")
    except ImportError:
        print("python-dotenv não instalado; seguindo sem load_dotenv.")

from graphrag.config import CHUNK_STRATEGY
from graphrag.indexing.load_and_chunk import run_load_and_chunk
from graphrag.indexing.extract_graph import run_extract_on_chunks
from graphrag.indexing.communities import run_communities
from graphrag.indexing.graph_links import link_claims_and_covariates_to_communities
from graphrag.indexing.reports import run_reports
from graphrag.indexing.embed import run_embed_all
from graphrag.store.neo4j_graph import get_neo4j_graph

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"GRAPHRAG_ROOT: {GRAPHRAG_ROOT}")

.env carregado de: /home/micael/mestrado/benchmarking-graphrag/graphrag-neo4j-langchain/.env
REPO_ROOT: /home/micael/mestrado/benchmarking-graphrag
GRAPHRAG_ROOT: /home/micael/mestrado/benchmarking-graphrag/graphrag-neo4j-langchain


In [2]:
# Parâmetros principais da indexação.
INPUT_DIR = REPO_ROOT / "pdfs_txt"
CHUNK_STRATEGY_OVERRIDE = None  # Use "tokens" ou "chars" para sobrescrever.

if not INPUT_DIR.exists() or not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Diretório de entrada não encontrado: {INPUT_DIR}")

print(f"Diretório de entrada: {INPUT_DIR}")
print(f"Estratégia de chunk padrão (config): {CHUNK_STRATEGY}")
print(f"Estratégia de chunk efetiva: {CHUNK_STRATEGY_OVERRIDE or CHUNK_STRATEGY}")

Diretório de entrada: /home/micael/mestrado/benchmarking-graphrag/pdfs_txt
Estratégia de chunk padrão (config): tokens
Estratégia de chunk efetiva: tokens


In [3]:
# Limpeza total dos dados do Neo4j antes da indexação.
graph = get_neo4j_graph()
driver = graph._driver

with driver.session() as session:
    before_counts = session.run(
        """
        MATCH (n)
        OPTIONAL MATCH ()-[r]-()
        RETURN count(DISTINCT n) AS nodes, count(DISTINCT r) AS relationships
        """
    ).single()

    nodes_before = before_counts["nodes"]
    rels_before = before_counts["relationships"]
    print(f"Antes da limpeza -> nós: {nodes_before}, relacionamentos: {rels_before}")

    if nodes_before > 0 or rels_before > 0:
        session.run("MATCH (n) DETACH DELETE n")
        print("Dados existentes removidos com sucesso.")
    else:
        print("Banco já estava vazio; nenhuma remoção necessária.")

    after_counts = session.run(
        """
        MATCH (n)
        OPTIONAL MATCH ()-[r]-()
        RETURN count(DISTINCT n) AS nodes, count(DISTINCT r) AS relationships
        """
    ).single()

    print(
        "Depois da limpeza -> "
        f"nós: {after_counts['nodes']}, relacionamentos: {after_counts['relationships']}"
    )

/home/micael/mestrado/benchmarking-graphrag/graphrag-neo4j-langchain/src/graphrag/store/neo4j_graph.py:14: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  _graph = Neo4jGraph(


Antes da limpeza -> nós: 4716, relacionamentos: 13022
Dados existentes removidos com sucesso.
Depois da limpeza -> nós: 0, relacionamentos: 0


In [7]:
# Indexação completa GraphRAG.
print("Iniciando load/chunk...")
chunk_records = run_load_and_chunk(INPUT_DIR, strategy=CHUNK_STRATEGY_OVERRIDE)
print(f"TextUnits criados: {len(chunk_records)}")

if not chunk_records:
    raise RuntimeError("Nenhum chunk foi criado. Verifique os documentos de entrada.")

print("Extraindo entidades e relacionamentos...")
run_extract_on_chunks(chunk_records)
print("Extração concluída.")

print("Detectando comunidades...")
communities = run_communities()
link_stats = link_claims_and_covariates_to_communities()
print(f"Comunidades L0 detectadas: {len(communities)}")
print(f"Vínculos Claims/Covariates: {link_stats}")

print("Gerando community reports...")
run_reports()
print("Community reports concluídos.")

print("Calculando embeddings...")
run_embed_all()
print("Embeddings concluídos.")

print("Pipeline de indexação finalizado com sucesso.")

Iniciando load/chunk...
TextUnits criados: 20
Extraindo entidades e relacionamentos...


KeyboardInterrupt: 

## Avaliação estrutural: Instantiated Class Ratio (ICR)

In [16]:
# Avaliação estrutural: Instantiated Class Ratio (ICR)
# ICR = (número de classes da ontologia com pelo menos 1 instância) / (número total de classes da ontologia)
# As classes ontológicas abaixo vêm do prompt de extração em extract_graph.py.

ontology_classes = [
    "Facility",
    "Organization",
    "Location",
    "Incident",
    "Material",
    "Equipment",
    "Process",
    "Personnel",
]

with get_neo4j_graph()._driver.session() as session:
    class_counts = {}
    for cls in ontology_classes:
        result = session.run(
            """
            MATCH (e:Entity)
            WHERE toLower(trim(coalesce(e.type, ""))) = toLower($class_name)
            RETURN count(e) AS c
            """,
            class_name=cls,
        ).single()
        class_counts[cls] = result["c"]

instantiated_classes = [cls for cls, count in class_counts.items() if count > 0]
total_classes = len(ontology_classes)
instantiated_count = len(instantiated_classes)
icr = instantiated_count / total_classes if total_classes else 0.0

print("Instantiated Class Ratio (ICR)")
print(f"- Classes da ontologia: {total_classes}")
print(f"- Classes instanciadas: {instantiated_count}")
print(f"- ICR: {icr:.4f} ({icr:.2%})")

print("\nDetalhamento por classe (Entity.type):")
for cls in ontology_classes:
    status = "instanciada" if class_counts[cls] > 0 else "sem instância"
    print(f"- {cls}: {class_counts[cls]} ({status})")

Instantiated Class Ratio (ICR)
- Classes da ontologia: 8
- Classes instanciadas: 8
- ICR: 1.0000 (100.00%)

Detalhamento por classe (Entity.type):
- Facility: 16 (instanciada)
- Organization: 24 (instanciada)
- Location: 6 (instanciada)
- Incident: 7 (instanciada)
- Material: 42 (instanciada)
- Equipment: 27 (instanciada)
- Process: 8 (instanciada)
- Personnel: 19 (instanciada)


In [23]:
# gold_graphs mockado
# Ground truth sintético para testar a métrica Graph-BERTScore (G-BS)
# sem depender da construção automática a partir do Neo4j.

gold_graphs = {
    "mock_tu_001": {
        "nodes": [
            {"name": "Aes Plant", "type": "Facility"},
            {"name": "Accurate Energetic Systems", "type": "Organization"},
            {"name": "McEwen, Tennessee", "type": "Location"},
            {"name": "March 2021 Explosion", "type": "Incident"},
            {"name": "RDX", "type": "Material"},
            {"name": "Melt-Pour Kettle", "type": "Equipment"},
            {"name": "Melt-Pour Operation", "type": "Process"},
            {"name": "Operators", "type": "Personnel"},
        ],
        "edges": [
            ("Aes Plant", "LOCATED_IN", "McEwen, Tennessee"),
            ("Aes Plant", "OPERATED_BY", "Accurate Energetic Systems"),
            ("March 2021 Explosion", "OCCURRED_AT", "Aes Plant"),
            ("Melt-Pour Operation", "USES_EQUIPMENT", "Melt-Pour Kettle"),
            ("March 2021 Explosion", "INVOLVES_MATERIAL", "RDX"),
        ],
    },
    "mock_tu_002": {
        "nodes": [
            {"name": "Building 2", "type": "Facility"},
            {"name": "US Chemical Safety Board", "type": "Organization"},
            {"name": "Main Production Site", "type": "Location"},
            {"name": "Secondary Fire", "type": "Incident"},
            {"name": "TNT", "type": "Material"},
            {"name": "Cooling System", "type": "Equipment"},
            {"name": "Quality Inspection", "type": "Process"},
            {"name": "Maintenance Team", "type": "Personnel"},
        ],
        "edges": [
            ("Building 2", "LOCATED_IN", "Main Production Site"),
            ("Secondary Fire", "OCCURRED_AT", "Building 2"),
            ("Secondary Fire", "INVOLVES_MATERIAL", "TNT"),
            ("Quality Inspection", "USES_EQUIPMENT", "Cooling System"),
            ("Secondary Fire", "RESULTED_IN", "Maintenance Team"),
        ],
    },
    "mock_tu_003": {
        "nodes": [
            {"name": "North Unit", "type": "Facility"},
            {"name": "Federal Investigators", "type": "Organization"},
            {"name": "Nashville Region", "type": "Location"},
            {"name": "Detonation Event", "type": "Incident"},
            {"name": "Propellant Mix", "type": "Material"},
            {"name": "Pressure Vessel", "type": "Equipment"},
            {"name": "Heating Cycle", "type": "Process"},
            {"name": "Line Supervisors", "type": "Personnel"},
        ],
        "edges": [
            ("North Unit", "LOCATED_IN", "Nashville Region"),
            ("Detonation Event", "OCCURRED_AT", "North Unit"),
            ("Detonation Event", "INVOLVES_MATERIAL", "Propellant Mix"),
            ("Heating Cycle", "USES_EQUIPMENT", "Pressure Vessel"),
            ("Detonation Event", "CONTRIBUTED_TO", "Line Supervisors"),
        ],
    },
}

print(f"gold_graphs mockado criado com {len(gold_graphs)} grafos.")
print(f"IDs disponíveis: {list(gold_graphs.keys())}")

gold_graphs mockado criado com 3 grafos.
IDs disponíveis: ['mock_tu_001', 'mock_tu_002', 'mock_tu_003']


In [21]:
!pip install bert-score

  Using cached triton-3.6.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached contourpy-1.3.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 33.2 MB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 37.0 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 49.4 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 50.8 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 36.3 MB/s  0:00:01

In [24]:
# Avaliação estrutural: Graph-BERTScore (G-BS)
# G-BS: Similaridade semântica entre grafo predito e grafo de referência,
# serializados em texto canônico por TextUnit.


def _norm_text(value: str) -> str:
    return " ".join(str(value or "").strip().lower().split())


def _canonical_node(node: dict) -> tuple[str, str]:
    return (_norm_text(node.get("type", "")), _norm_text(node.get("name", "")))


def _canonical_edge(edge: tuple[str, str, str]) -> tuple[str, str, str]:
    src, rel, dst = edge
    return (_norm_text(src), _norm_text(rel), _norm_text(dst))


def _serialize_graph(graph_obj: dict) -> str:
    nodes = sorted(graph_obj.get("nodes", []))
    edges = sorted(graph_obj.get("edges", []))

    node_lines = [f"NODE | type={t} | name={n}" for t, n in nodes]
    edge_lines = [f"EDGE | src={s} | rel={r} | dst={d}" for s, r, d in edges]

    return "\n".join(node_lines + edge_lines).strip()


if "gold_graphs" not in globals():
    gold_graphs = {}

with get_neo4j_graph()._driver.session() as session:
    rows = session.run(
        """
        MATCH (t:TextUnit)-[:MENTIONS]->(e:Entity)
        OPTIONAL MATCH (t)-[:MENTIONS]->(s:Entity)-[r:RELATES_TO]->(d:Entity)<-[:MENTIONS]-(t)
        RETURN t.id AS tu_id,
               collect(DISTINCT {name: e.name, type: e.type}) AS nodes,
               collect(DISTINCT [s.name, r.type, d.name]) AS edges
        """
    )

    pred_graphs = {}
    for row in rows:
        tu_id = row["tu_id"]
        pred_nodes = {_canonical_node(n) for n in (row["nodes"] or []) if n and n.get("name")}
        pred_edges = {
            _canonical_edge((e[0], e[1], e[2]))
            for e in (row["edges"] or [])
            if e and len(e) == 3 and e[0] and e[1] and e[2]
        }
        pred_graphs[tu_id] = {"nodes": pred_nodes, "edges": pred_edges}

if not gold_graphs:
    print("Graph-BERTScore (G-BS): sem ground truth.")
    print("Defina `gold_graphs` no formato indicado e reexecute esta célula.")
else:
    try:
        from bert_score import score as bertscore_score
    except ImportError as exc:
        raise ImportError(
            "Pacote `bert-score` não encontrado. Instale com: pip install bert-score"
        ) from exc

    # Canonicaliza ground truth.
    gold_canonical = {}
    for tu_id, g in gold_graphs.items():
        nodes = {_canonical_node(n) for n in g.get("nodes", []) if n and n.get("name")}
        edges = {_canonical_edge(e) for e in g.get("edges", []) if e and len(e) == 3}
        gold_canonical[tu_id] = {"nodes": nodes, "edges": edges}

    # Avalia somente interseção de tu_ids entre predito e referência.
    common_tu_ids = sorted(set(pred_graphs).intersection(gold_canonical))

    if not common_tu_ids:
        print("Graph-BERTScore (G-BS): não há tu_ids em comum entre predição e gold_graphs.")
    else:
        cands = [_serialize_graph(pred_graphs[tu_id]) for tu_id in common_tu_ids]
        refs = [_serialize_graph(gold_canonical[tu_id]) for tu_id in common_tu_ids]

        # Modelo leve por padrão para reduzir custo; ajuste se necessário.
        P, R, F1 = bertscore_score(
            cands,
            refs,
            lang="en",
            model_type="distilbert-base-uncased",
            verbose=False,
        )

        g_bs_precision = float(P.mean().item())
        g_bs_recall = float(R.mean().item())
        g_bs_f1 = float(F1.mean().item())

        print("Graph-BERTScore (G-BS)")
        print(f"- Grafos avaliados (tu_ids em comum): {len(common_tu_ids)}")
        print(f"- G-BS Precision (média): {g_bs_precision:.4f}")
        print(f"- G-BS Recall (média): {g_bs_recall:.4f}")
        print(f"- G-BS F1 (média): {g_bs_f1:.4f} ({g_bs_f1:.2%})")

        # Opcional: inspeção por TextUnit.
        g_bs_by_tu = {
            tu_id: {
                "precision": float(P[i].item()),
                "recall": float(R[i].item()),
                "f1": float(F1[i].item()),
            }
            for i, tu_id in enumerate(common_tu_ids)
        }

Graph-BERTScore (G-BS): não há tu_ids em comum entre predição e gold_graphs.


In [9]:
# Verificação final de nós (labels) e arestas (tipos de relacionamento).
with get_neo4j_graph()._driver.session() as session:
    node_labels = [
        "Document",
        "TextUnit",
        "Entity",
        "Community",
        "CommunityReport",
        "Claim",
        "Covariate",
    ]

    print("Contagem de nós por label:")
    for label in node_labels:
        result = session.run(f"MATCH (n:{label}) RETURN count(n) AS c").single()
        print(f"- {label}: {result['c']}")

    relationship_types = [
        "HAS_CHUNK",
        "MENTIONS",
        "RELATES_TO",
        "IN_COMMUNITY",
        "HAS_SUBCOMMUNITY",
        "HAS_FINDING",
        "EVIDENCE_FOR",
        "HAS_CLAIM",
        "HAS_COVARIATE",
    ]

    print("\nContagem de arestas por tipo:")
    for rel_type in relationship_types:
        result = session.run(f"MATCH ()-[r:{rel_type}]->() RETURN count(r) AS c").single()
        print(f"- {rel_type}: {result['c']}")

    total = session.run(
        """
        MATCH (n)
        OPTIONAL MATCH ()-[r]-()
        RETURN count(DISTINCT n) AS nodes, count(DISTINCT r) AS relationships
        """
    ).single()

    print("\nTotais gerais:")
    print(f"- Nós: {total['nodes']}")
    print(f"- Relacionamentos: {total['relationships']}")

print("Validação final concluída.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_SUBCOMMUNITY` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 12, 'line': 1, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:HAS_SUBCOMMUNITY]->() RETURN count(r) AS c'


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_FINDING` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 12, 'line': 1, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:HAS_FINDING]->() RETURN count(r) AS c'


Contagem de nós por label:
- Document: 1
- TextUnit: 20
- Entity: 171
- Community: 106
- CommunityReport: 106
- Claim: 181
- Covariate: 104

Contagem de arestas por tipo:
- HAS_CHUNK: 20
- MENTIONS: 227
- RELATES_TO: 169
- IN_COMMUNITY: 171
- HAS_SUBCOMMUNITY: 0
- HAS_FINDING: 0
- EVIDENCE_FOR: 285
- HAS_CLAIM: 179
- HAS_COVARIATE: 104

Totais gerais:
- Nós: 689
- Relacionamentos: 1617
Validação final concluída.
